# 06 — Deep Learning Literature Review

**Project**: Predictive Sales Analytics Engine — Repeat Purchase Prediction  
**Phase**: 2 (Deep Learning)  
**Objective**: Review modern deep learning approaches for tabular classification and justify our architecture choice.

---

## 1. Problem Context

Our task is **binary classification on tabular data**: predicting whether a customer will make a repeat purchase within 180 days using 16 numeric features (delivery metrics, payment data, review statistics, etc.).

In Phase 1, traditional ML baselines (Logistic Regression, Random Forest) achieved modest performance (best PR-AUC = 0.023, ROC-AUC = 0.57), limited by extreme class imbalance (1.8% positive rate) and weak individual feature correlations.

Phase 2 investigates whether deep learning can improve on these baselines by learning non-linear feature interactions that tree-based and linear models may miss.

## 2. Deep Learning for Tabular Data — Recent Literature

### 2.1 Can Deep Learning Beat Trees on Tabular Data?

**Grinsztajn, Oyallon & Varoquaux (2022)** — *"Why do tree-based models still outperform deep learning on typical tabular data?"* (NeurIPS 2022)

This benchmark study compared gradient-boosted decision trees (GBDTs) against deep learning models across 45 tabular datasets. Key findings:
- Tree-based models (XGBoost, Random Forest) consistently outperform neural networks on medium-sized tabular datasets.
- Neural networks struggle with **irregular feature distributions** and **uninformative features** — both present in our dataset.
- However, the gap narrows when neural networks are **properly regularized and tuned**.
- DL can still add value when there are **complex feature interactions** that trees cannot capture efficiently, when **class imbalance** is present (neural networks handle weighted loss functions more smoothly), or when the dataset benefits from **representation learning**.

**Relevance to our project**: Our dataset has 52K rows with skewed distributions, weak features, and extreme class imbalance (1.8% positive) — a setting where trees typically dominate. We should not expect the DL model to drastically outperform Random Forest, but rather aim for competitive performance with the benefit of learning richer feature interactions.

---

### 2.2 Well-Tuned Simple Networks

**Kadra, Lindauer, Hutter & Grabocka (2021)** — *"Well-tuned Simple Nets Excel on Tabular Datasets"* (NeurIPS 2021)

This paper demonstrated that a simple Multi-Layer Perceptron (MLP), when combined with a **cocktail of regularization techniques** (Dropout, Weight Decay, Batch Normalization, early stopping), can match or exceed more complex deep learning architectures on tabular data.

Their key insight: **architecture complexity matters less than regularization strategy** for tabular data. A 2-3 layer MLP with proper regularization outperformed TabNet, NODE, and other specialized architectures on many benchmarks.

**Relevance to our project**: This directly motivates our architecture choice — a well-regularized MLP rather than a complex specialized model. The paper validates that simplicity + proper regularization is the right approach for our dataset size.

---

### 2.3 Revisiting Deep Learning Models for Tabular Data

**Gorishniy, Rubachev, Khrulkov & Babenko (2021)** — *"Revisiting Deep Learning Models for Tabular Data"* (NeurIPS 2021)

This study systematically compared ResNet-like MLPs, FT-Transformer (Feature Tokenizer + Transformer), and traditional models. Findings:
- A **ResNet-like MLP** (MLP with skip connections) is a strong baseline that is competitive with GBDTs.
- FT-Transformer works well but requires more compute and careful tuning.
- Simple MLPs underperform unless given proper depth and regularization.
- MLPs with a **funnel-shaped architecture** (wide → narrow layers) with BatchNorm outperform flat architectures.

**Relevance to our project**: Confirms that an MLP with BatchNorm and a decreasing-width design is a sound choice. We adopt the funnel-shaped MLP (64 → 32 → 1) with BatchNorm as recommended.

## 3. Comparison of Approaches

| Approach | Strengths | Weaknesses | Fit for Our Task |
|----------|-----------|------------|------------------|
| **GBDT (XGBoost/RF)** | Best overall on tabular data; handles irregular distributions well | Cannot learn smooth feature interactions; limited by greedy splits | Strong baseline (Phase 1) |
| **MLP (our choice)** | Learns non-linear feature interactions; flexible loss weighting; simple to implement and explain | Needs careful regularization; sensitive to feature scaling | Good — simple, justified, proper regularization compensates |
| **Transformer (FT-Transformer)** | Captures feature-feature attention; SOTA on some tabular benchmarks | Over-parameterized for 52K rows; slow training; hard to explain | Over-kill — too complex for dataset size |
| **TabNet** | Built-in feature selection; interpretable attention masks | Unstable training; marginal gains over tuned MLP | Unnecessary complexity |
| **LSTM/RNN** | Excellent for sequential data | Our data is not sequential — each row is an independent order | Wrong architecture for this data type |

## 4. Our Approach and Gap Analysis

### What existing models struggle with:
- **Tree-based models** make axis-aligned splits — they cannot learn smooth diagonal decision boundaries in feature space. For example, the interaction between `delivery_days` and `review_score` may have a non-linear combined effect that individual splits cannot capture efficiently.
- **Linear models** assume additive feature effects — they miss interactions entirely unless manually engineered.

### What our DL approach adds:
- **Automatic interaction learning**: Each hidden layer neuron computes a weighted combination of all input features, then applies a non-linear activation. This allows the MLP to learn arbitrary feature interactions without manual engineering.
- **Smooth probability calibration**: The sigmoid output layer produces continuous probabilities, and weighted BCE loss directly optimizes for the class imbalance.
- **Regularization cocktail**: Following Kadra et al. (2021), we combine Dropout + BatchNorm + Weight Decay + Early Stopping to prevent overfitting on our small dataset.

### Expected outcome:
Based on the literature (Grinsztajn et al., 2022), we expect the MLP to perform **comparably to Random Forest** (within ±0.005 PR-AUC), with the primary value being:
1. Demonstrating that DL can work on this tabular task when properly configured.
2. Providing a foundation for future extensions (e.g., adding text embeddings, entity embeddings for categorical features).

## 5. References

1. Grinsztajn, L., Oyallon, E., & Varoquaux, G. (2022). Why do tree-based models still outperform deep learning on typical tabular data? *Advances in Neural Information Processing Systems (NeurIPS)*, 35. Available at: [arXiv:2207.08815](https://arxiv.org/abs/2207.08815)

2. Kadra, A., Lindauer, M., Hutter, F., & Grabocka, J. (2021). Well-tuned Simple Nets Excel on Tabular Datasets. *Advances in Neural Information Processing Systems (NeurIPS)*, 34. Available at: [arXiv:2106.11189](https://arxiv.org/abs/2106.11189)

3. Gorishniy, Y., Rubachev, I., Khrulkov, V., & Babenko, A. (2021). Revisiting Deep Learning Models for Tabular Data. *Advances in Neural Information Processing Systems (NeurIPS)*, 34. Available at: [arXiv:2106.11959](https://arxiv.org/abs/2106.11959)

## 6. How Each Reference Shaped Our Design Decisions

The table below maps every major decision in our Phase 2 implementation back to the specific paper that motivated it.

| Design Decision | Source Paper | What We Learned | How It Influenced Our Code |
|----------------|-------------|-----------------|---------------------------|
| **Set realistic expectations vs trees** | Grinsztajn et al. (2022) | Trees dominate on medium-sized tabular data with irregular distributions — exactly our setting (52K rows, skewed features) | We framed the DL model as "competitive with" rather than "superior to" our Phase 1 baselines. Used weighted BCE loss for class imbalance, as DL handles this more smoothly than trees |
| **Chose MLP over Transformer/TabNet** | Kadra et al. (2021) | A simple MLP with proper regularization matches or beats complex architectures on tabular benchmarks | We implemented a 2-layer MLP instead of TabNet or FT-Transformer, saving complexity without sacrificing performance |
| **Regularization cocktail (Dropout + BN + WD + ES)** | Kadra et al. (2021) | Combining multiple regularization techniques is more effective than any single technique alone | We applied Dropout(0.3), BatchNorm, Weight Decay(1e-4), and Early Stopping(patience=15) together |
| **Funnel-shaped architecture (64→32→1)** | Gorishniy et al. (2021) | ResNet-like MLPs with decreasing layer widths are a strong baseline for tabular tasks | We adopted the wide-to-narrow funnel shape with BatchNorm, as their ablation showed this outperforms flat architectures |
| **AdamW + ReLU choices** | Gorishniy et al. (2021) | Their MLP baselines used AdamW with ReLU activations as the standard configuration | We followed the same optimizer and activation setup for consistency with proven tabular DL practice |

> **Key takeaway**: Every architectural and training decision traces back to a peer-reviewed NeurIPS publication. We assembled components from the literature based on our specific dataset characteristics (small, tabular, imbalanced).